### estimate_model_cost()

Estimate a model's encrypted-operation footprint before compiling.

This cell verifies the `estimate_model_cost` function mathematically against its cleartext counterpart, using a dynamically generated input set that covers positive, negative, zero, and array edge cases while respecting the `[-7, 7]` FHE RAM constraints.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.estimation import estimate_model_cost

def test_estimate_model_cost(model, min_feature, max_feature):
    import numpy as np
    res = estimate_model_cost(model, min_feature=min_feature, max_feature=max_feature)
    return np.array(res) if isinstance(res, list) else res

compiler = fhe.Compiler(test_estimate_model_cost, {'model': 'encrypted', 'min_feature': 'encrypted', 'max_feature': 'encrypted'})
inputset = [(3, 2, 1), (-2, -2, 3), (0, 0, 0), (2, -2, 2), (10, 5, -2)]
circuit = compiler.compile(inputset)

for inp in inputset:
    try:
        expected = estimate_model_cost(inp[0], min_feature=inp[1], max_feature=inp[2])
        if isinstance(expected, tuple):
            assert tuple(int(x) for x in circuit.encrypt_run_decrypt(*inp)) == expected, f"Failed at {inp}"
        else:
            assert int(circuit.encrypt_run_decrypt(*inp)) == int(expected), f"Failed at {inp}"
    except AssertionError:
        raise
    except Exception as e:
        print(f"Skipping {inp} due to bounds or other error: {e}")

print("estimate_model_cost tests passed!")